# 03 Pause for human approval
Add the write-capable refund tool while keeping a person in control of the action. The middleware pauses before that tool executes, and the code guardrails still apply after approval. All orders and refunds are fictional, and the handbook search only reads Session 10's `heading-800` namespace. Create `.env` with both keys before using Google or Pinecone; local order/refund checks can run without them.

Find the repository whether Jupyter starts here or in the notebooks folder. Keep the two Google model names visible, and import the shared implementation so all notebooks use the same tool behavior.

In [ ]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
EMBEDDING_MODEL = "google_genai:gemini-embedding-001"
CHAT_MODEL = "google_genai:gemini-3.7-flash"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import agent as lab
load_dotenv(ROOT / ".env")

Check that the required files and key names are present without printing secrets. The log labels show observable tool choices and results, not hidden model reasoning; LangSmith tracing is disabled by the shared module.

In [ ]:
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", force=True)
missing = []
for name in ("GOOGLE_API_KEY", "PINECONE_API_KEY"):
    if not os.getenv(name):
        missing.append(name)
if missing:
    print("Add these keys to .env:", ", ".join(missing))
else:
    print("Both keys are present; live requests check access.")
assert lab.ORDERS_PATH.is_file(), "Restore data/orders.csv before continuing."

The middleware matches the tool name `issue_refund`, while `InMemorySaver` preserves the paused messages and pending action. The system prompt is identical to the read-only agent; the new behavior comes from tool availability and middleware, not a prompt saying “please ask”.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from uuid import uuid4
approval = HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True})
agent = create_agent(model=lab.get_model(), tools=lab.TOOLS,
    system_prompt=lab.SYSTEM_PROMPT, middleware=[approval],
    checkpointer=InMemorySaver())

Run ticket 1 in a fresh thread and inspect any proposed refund before it is executed. `recursion_limit` bounds graph execution, and a unique `thread_id` keeps this complaint's checkpoint separate from other runs.

In [ ]:
message = "My stove smells of gas. Order T-1042. I want my money back."
config = {"recursion_limit": 10, "configurable": {"thread_id": str(uuid4())}}
result = {"messages": []}
try:
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config=config)
    for pause in result.get("__interrupt__", []):
        print(pause.value)
except Exception:
    print("Request failed: check keys, index readiness, or the step limit.")

When prompted below, choose **approve** for this first run and verify the exact order and amount. `review_actions` reads human input, records the decision, and grants a one-use receipt; `Command(resume=...)` sends the matching ordered decisions back to the paused graph.

In [ ]:
try:
    while result.get("__interrupt__"):
        actions = result["__interrupt__"][0].value["action_requests"]
        decision = lab.review_actions(actions)
        result = agent.invoke(Command(resume=decision), config=config)
    for m in result["messages"]:
        m.pretty_print()
except Exception:
    print("Request failed: check keys/limits and inspect the trace before retrying.")
finally:
    lab.APPROVALS.clear()

Rerun the same ticket in a new thread, and choose **reject** if it proposes a refund. The shared runner handles the same interrupt/resume process with local trace logs; rejecting never executes the tool, and even approving again cannot bypass the duplicate-refund guard.

In [ ]:
second_result = lab.run_agent(agent, message)
print("Model/tool updates:", second_result["steps"])

Compare the tool result and final reply across the two runs rather than assuming the model always chooses the same tool sequence. If it does not propose a refund, there is no approval prompt; inspect the actual trace and the cited policy.